In [3]:
#0.2017870112506833

In [1]:
import os
import json
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ===== 評価式ベースの損失関数 =====
class SubaruEvalLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, preds, targets):
        preds = preds[:, 19:]
        targets = targets[:, 19:]
        diff = torch.abs(preds - targets)
        denom = 0.07 * targets + 3
        err = diff / denom
        err = torch.clamp(err, max=1.0)
        return err.mean()

# ===== モデル定義 =====
class LSTM260D_v3(nn.Module):
    def __init__(self, input_size=13, hidden_size=384, num_layers=3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )
        self.fc = nn.Sequential(
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        pooled = out.mean(dim=1)
        return self.fc(pooled).squeeze(1)

# ===== Dataset定義 =====
class RelativeSpeedDataset260D(Dataset):
    def __init__(self, annot_root, distance_json_path, max_items=None):
        self.items = []

        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue

            sid = fname.replace(".json", "")
            if sid not in self.distances:
                continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)

            seq = ann.get("sequence", [])
            if len(seq) < 20:
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            tgt = np.array([f['TgtSpeed_ref'] for f in seq], dtype=np.float32)
            keys = [f"frame_{i+1:05d}" for i in range(len(seq))]
            dist = np.array([self.distances[sid].get(k, np.nan) for k in keys], dtype=np.float32)

            if len(dist) < 20:
                continue

            def smooth(x, w):
                if len(x) < w:
                    return np.zeros_like(x)
                return np.convolve(x, np.ones(w)/w, mode='same')

            for i in range(len(seq) - 19):
                if max_items and len(self.items) >= max_items:
                    return

                d = dist[i:i+20]
                o = own[i:i+20]
                t = tgt[i:i+20]

                if np.any(np.isnan(d)) or np.any(np.isnan(o)) or np.any(np.isnan(t)):
                    continue

                rel_speed = t - o
                own_acc = np.gradient(o)
                d1 = np.gradient(d)
                d2 = np.gradient(d1)

                f3 = smooth(d, 3)
                f5 = smooth(d, 5)
                f7 = smooth(d, 7)
                f11 = smooth(d, 11)
                f11_d1 = np.gradient(f11) if len(f11) >= 3 else np.zeros_like(f11)

                try:
                    feat = np.stack([
                        d[:20], o[:20], own_acc[:20], d1[:20], d2[:20],
                        f3[:20], f5[:20], f7[:20], f11[:20], f11_d1[:20],
                        f3[:20] * d1[:20], f11[:20] - f5[:20], np.abs(d1[:20])
                    ], axis=1)
                except:
                    continue

                if feat.shape != (20, 13):
                    continue

                target = np.mean(rel_speed)
                self.items.append((feat.astype(np.float32), target, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, tgt, sid = self.items[idx]
        return feat, tgt, sid

# ===== 学習ループ（評価指標最小化用） =====
def train_eval_loss(dataset, save_path="model_lstm260d_v3_eval.pth"):
    scenes = sorted(set([item[-1] for item in dataset.items]))
    train_scenes, val_scenes = train_test_split(scenes, test_size=0.2, random_state=42)
    train_ds = [item for item in dataset.items if item[-1] in train_scenes]
    val_ds = [item for item in dataset.items if item[-1] in val_scenes]

    def collate_fn(batch):
        feats, tgts, sids = zip(*batch)
        feats = [torch.tensor(f, dtype=torch.float32) for f in feats]
        tgts = [torch.tensor([t]*20, dtype=torch.float32) for t in tgts]  # 20フレームに展開
        return torch.stack(feats), torch.stack(tgts), list(sids)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = LSTM260D_v3().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.4, patience=4)
    criterion = SubaruEvalLoss()

    best_val_loss = float('inf')
    patience = 10
    counter = 0

    for epoch in range(50):
        model.train()
        total_train_loss = 0
        for feats, tgts, _ in tqdm(train_loader, desc=f"[Train Epoch {epoch+1}]"):
            feats, tgts = feats.to(device), tgts.to(device)
            pred = model(feats).unsqueeze(1).repeat(1, 20)  # 出力を20フレーム分に複製
            loss = criterion(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * feats.size(0)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for feats, tgts, _ in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                pred = model(feats).unsqueeze(1).repeat(1, 20)
                loss = criterion(pred, tgts)
                total_val_loss += loss.item() * feats.size(0)

        train_loss = total_train_loss / len(train_ds)
        val_loss = total_val_loss / len(val_ds)
        scheduler.step(val_loss)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ モデル保存: {save_path}（val_loss={val_loss:.4f}）")
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print("🛑 Early Stopping")
                break

# ===== 実行部 =====
if __name__ == "__main__":
    dataset = RelativeSpeedDataset260D(
        annot_root="./train_annotations",
        distance_json_path="../distance_ref_data.json",
        max_items=40000
    )
    print(f"✅ dataset loaded: {len(dataset)} samples")
    train_eval_loss(dataset, save_path="model_lstm260d_v3_eval.pth")

✅ dataset loaded: 40000 samples


[Train Epoch 1]: 100%|██████████| 499/499 [00:05<00:00, 87.89it/s]


Epoch 1 | Train Loss: 0.3432 | Val Loss: 0.3314
✅ モデル保存: model_lstm260d_v3_eval.pth（val_loss=0.3314）


[Train Epoch 2]: 100%|██████████| 499/499 [00:05<00:00, 96.51it/s]


Epoch 2 | Train Loss: 0.3011 | Val Loss: 0.3273
✅ モデル保存: model_lstm260d_v3_eval.pth（val_loss=0.3273）


...

Epoch 48 | Train Loss: 0.0846 | Val Loss: 0.0422


[Train Epoch 49]: 100%|██████████| 499/499 [00:05<00:00, 85.60it/s]


Epoch 49 | Train Loss: 0.0834 | Val Loss: 0.0444


[Train Epoch 50]: 100%|██████████| 499/499 [00:05<00:00, 85.50it/s]


Epoch 50 | Train Loss: 0.0844 | Val Loss: 0.0504


In [2]:
import os
import json
import numpy as np
import joblib
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from tqdm import tqdm

# -------- モデル定義 --------
class LSTM260D_v3(nn.Module):
    def __init__(self, input_size=13, hidden_size=384, num_layers=3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )
        self.fc = nn.Sequential(
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        pooled = out.mean(dim=1)
        return self.fc(pooled).squeeze(1)

# -------- 推論用 Dataset --------
class InferenceDataset260D(Dataset):
    def __init__(self, annot_root, distance_json_path):
        self.items = []
        self.seq_lens = {}

        with open(distance_json_path, encoding='utf-8') as f:
            self.distances_raw = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue

            sid = fname.replace(".json", "")
            if sid not in self.distances_raw:
                continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)

            seq = ann.get("sequence", [])
            self.seq_lens[sid] = len(seq)
            if len(seq) < 20:
                continue

            own = np.array([f["OwnSpeed"] for f in seq], dtype=np.float32)
            keys = [f"frame_{i+1:05d}" for i in range(len(seq))]
            dist = np.array([self.distances_raw[sid].get(k, np.nan) for k in keys], dtype=np.float32)
            dist = self.fill_missing_linear(dist)

            def smooth(x, w):
                return np.convolve(x, np.ones(w) / w, mode='same') if len(x) >= w else np.zeros_like(x)

            for i in range(len(seq) - 19):
                d = dist[i:i+20]
                o = own[i:i+20]
                if np.any(np.isnan(d)) or np.any(np.isnan(o)):
                    continue

                try:
                    own_acc = np.gradient(o)
                    d1 = np.gradient(d)
                    d2 = np.gradient(d1)
                    f3 = smooth(d, 3)
                    f5 = smooth(d, 5)
                    f7 = smooth(d, 7)
                    f11 = smooth(d, 11)
                    f11_d1 = np.gradient(f11)

                    feat = np.stack([
                        d[:20], o[:20], own_acc[:20], d1[:20], d2[:20],
                        f3[:20], f5[:20], f7[:20], f11[:20], f11_d1[:20],
                        f3[:20] * d1[:20], f11[:20] - f5[:20], np.abs(d1[:20])
                    ], axis=1)
                except:
                    continue

                if feat.shape != (20, 13):
                    continue

                own_last = o[-1]  # ✅ 最終フレームの自車速度を使う
                self.items.append((torch.tensor(feat, dtype=torch.float32), own_last, sid, i))

    def fill_missing_linear(self, arr):
        arr = np.array(arr, dtype=np.float32)
        if not np.any(np.isnan(arr)):
            return arr
        x = np.arange(len(arr))
        valid = ~np.isnan(arr)
        if valid.sum() < 2:
            return np.zeros_like(arr)
        arr[~valid] = np.interp(x[~valid], x[valid], arr[valid])
        return arr

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

# -------- 推論関数 --------
def predict_with_lstm260d(
    model_path,
    annot_root,
    distance_json_path,
    save_path="submission.json"
):
    dataset = InferenceDataset260D(annot_root, distance_json_path)
    loader = DataLoader(dataset, batch_size=64, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = LSTM260D_v3().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    from collections import defaultdict
    raw_preds = defaultdict(list)

    with torch.no_grad():
        for feats, own_speeds, sids, frame_idxs in tqdm(loader):
            feats = feats.to(device)
            own_speeds_np = own_speeds.numpy()
            preds = model(feats).cpu().numpy()
            abs_preds = preds + own_speeds_np

            for sid, frame_idx, pred in zip(sids, frame_idxs, abs_preds):
                raw_preds[sid].append((frame_idx + 19, float(round(pred, 3))))

    submission = {}
    for sid, pairs in raw_preds.items():
        pairs.sort()
        seq_len = dataset.seq_lens.get(sid, max(f for f, _ in pairs) + 1)
        pred_list = [0.0] * seq_len
        for idx, val in pairs:
            if idx < seq_len:
                pred_list[idx] = val
        for i in range(1, seq_len):
            if pred_list[i] == 0.0:
                pred_list[i] = pred_list[i - 1]
        submission[sid] = pred_list

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, indent=2, ensure_ascii=False)

    print(f"✅ 完了: {save_path} に保存（scene数: {len(submission)}）")

# -------- 実行 --------
if __name__ == "__main__":
    predict_with_lstm260d(
        model_path="model_lstm260d_v3_eval.pth",
        annot_root="../test2/test_annotations",
        distance_json_path="../testdistance/kernel/test_spline_smoothed.json",
        save_path="submission.json"
    )

100%|██████████| 396/396 [00:01<00:00, 216.10it/s]


✅ 完了: submission.json に保存（scene数: 239）
